# Dengue Data with weather

In [34]:
import pandas as pd
import numpy as np
import os

In [35]:
df_dengue = pd.read_csv('silver/dengue_city_geolocated.csv')
df_dengue = df_dengue[df_dengue['ANO'] == 2023]
df_dengue['date'] = pd.to_datetime(df_dengue['date'])
# week is in format dd-mm-yyyy
df_dengue['week'] = pd.to_datetime(df_dengue['week'], format='%d-%m-%Y')
df_dengue

,SEMANA,ANO,country,state,city,count,idx_city,Latitude,Longitude,date,week
131554,1,2023,aruba,exterior,exterior_aruba,1,ARUBA_EXTERIOR_EXTERIOR_ARUBA,12.501363,-69.961848,2023-01-08,2023-01-08
131555,1,2023,brasil,exterior,exterior_brasil,1,BRASIL_EXTERIOR_EXTERIOR_BRASIL,-10.333333,-53.200000,2023-01-08,2023-01-08
131556,1,2023,colombia,amazonas,leticia,7,COLOMBIA_AMAZONAS_LETICIA,-4.212921,-69.942596,2023-01-08,2023-01-08
131557,1,2023,colombia,antioquia,apartado,7,COLOMBIA_ANTIOQUIA_APARTADO,7.884901,-76.622746,2023-01-08,2023-01-08
131558,1,2023,colombia,antioquia,caracoli,1,COLOMBIA_ANTIOQUIA_CARACOLI,6.409276,-74.756698,2023-01-08,2023-01-08
...,...,...,...,...,...,...,...,...,...,...,...
147043,52,2023,colombia,vaupes,mitu,8,COLOMBIA_VAUPES_MITU,1.253850,-70.234558,2023-12-31,2023-12-31
147044,52,2023,colombia,vichada,puerto carreno,1,COLOMBIA_VICHADA_PUERTO CARREÑO,6.190923,-67.484189,2023-12-31,2023-12-31
147045,52,2023,comoras,exterior,exterior_comoras,1,COMORAS_EXTERIOR_EXTERIOR_COMORAS,-12.204518,44.283296,2023-12-31,2023-12-31
147046,52,2023,haiti,exterior,exterior_haiti,1,HAITÍ_EXTERIOR_EXTERIOR_HAITÍ,19.139995,-72.357097,2023-12-31,2023-12-31


In [36]:
df_dengue = df_dengue[df_dengue['country'] == 'colombia']
df_dengue

,SEMANA,ANO,country,state,city,count,idx_city,Latitude,Longitude,date,week
131556,1,2023,colombia,amazonas,leticia,7,COLOMBIA_AMAZONAS_LETICIA,-4.212921,-69.942596,2023-01-08,2023-01-08
131557,1,2023,colombia,antioquia,apartado,7,COLOMBIA_ANTIOQUIA_APARTADO,7.884901,-76.622746,2023-01-08,2023-01-08
131558,1,2023,colombia,antioquia,caracoli,1,COLOMBIA_ANTIOQUIA_CARACOLI,6.409276,-74.756698,2023-01-08,2023-01-08
131559,1,2023,colombia,antioquia,carepa,2,COLOMBIA_ANTIOQUIA_CAREPA,7.798452,-76.746039,2023-01-08,2023-01-08
131560,1,2023,colombia,antioquia,caucasia,3,COLOMBIA_ANTIOQUIA_CAUCASIA,7.987758,-75.198374,2023-01-08,2023-01-08
...,...,...,...,...,...,...,...,...,...,...,...
147040,52,2023,colombia,valle,vijes,2,COLOMBIA_VALLE_VIJES,3.700400,-76.442888,2023-12-31,2023-12-31
147041,52,2023,colombia,valle,yumbo,66,COLOMBIA_VALLE_YUMBO,3.583466,-76.495222,2023-12-31,2023-12-31
147042,52,2023,colombia,valle,zarzal,7,COLOMBIA_VALLE_ZARZAL,4.393943,-76.070647,2023-12-31,2023-12-31
147043,52,2023,colombia,vaupes,mitu,8,COLOMBIA_VAUPES_MITU,1.253850,-70.234558,2023-12-31,2023-12-31


In [37]:
df_dengue_s = df_dengue[['idx_city', 'week','count','Latitude', 'Longitude']].copy()
df_dengue_s['week'] = df_dengue_s['week'].dt.to_period('W').apply(lambda r: r.start_time)

In [38]:
df_dengue_s.reset_index(drop=True, inplace=True)

In [41]:
df_cities =  df_dengue[['idx_city','Latitude', 'Longitude']].copy()
df_cities.drop_duplicates(subset=['idx_city'], inplace=True)

In [42]:
df_cities.head()

,idx_city,Latitude,Longitude
131556,COLOMBIA_AMAZONAS_LETICIA,-4.212921,-69.942596
131557,COLOMBIA_ANTIOQUIA_APARTADO,7.884901,-76.622746
131558,COLOMBIA_ANTIOQUIA_CARACOLI,6.409276,-74.756698
131559,COLOMBIA_ANTIOQUIA_CAREPA,7.798452,-76.746039
131560,COLOMBIA_ANTIOQUIA_CAUCASIA,7.987758,-75.198374


# Elevation

In [43]:
df_elevation = pd.read_csv('meteo_colombia/cities_dengue_elevation.csv')
df_elevation

,idx_city,elevation
0,COLOMBIA_ANTIOQUIA_CARACOLI,616.0
1,COLOMBIA_ANTIOQUIA_CAUCASIA,59.0
2,COLOMBIA_ANTIOQUIA_EL BAGRE,518.0
3,COLOMBIA_ANTIOQUIA_MUTATA,133.0
4,COLOMBIA_ANTIOQUIA_NECHI,31.0
...,...,...
783,COLOMBIA_CAUCA_CAJIBIO,1828.0
784,COLOMBIA_ANTIOQUIA_RIONEGRO,2091.0
785,COLOMBIA_CAUCA_SILVIA,2495.0
786,COLOMBIA_QUINDIO_BUENAVISTA,1480.0


# Create rows of zeros

In [44]:
df_dengue_s.head()

,idx_city,week,count,Latitude,Longitude
0,COLOMBIA_AMAZONAS_LETICIA,2023-01-02,7,-4.212921,-69.942596
1,COLOMBIA_ANTIOQUIA_APARTADO,2023-01-02,7,7.884901,-76.622746
2,COLOMBIA_ANTIOQUIA_CARACOLI,2023-01-02,1,6.409276,-74.756698
3,COLOMBIA_ANTIOQUIA_CAREPA,2023-01-02,2,7.798452,-76.746039
4,COLOMBIA_ANTIOQUIA_CAUCASIA,2023-01-02,3,7.987758,-75.198374


In [45]:
unique_dates = df_dengue_s['week'].unique()
unique_idx = df_dengue_s['idx_city'].unique()

In [46]:
# create each combination of dates and indices
# Create all combinations of dates and city indices
df_complete = pd.MultiIndex.from_product(
    [unique_idx, unique_dates],
    names=['idx_city', 'week']
).to_frame(index=False)



In [31]:
df_dengue_l = df_complete.merge(df_dengue_s, how = 'left', on = ['idx_city', 'week'])
df_dengue_l['count'] = df_dengue_l['count'].fillna(0)
df_dengue_m = df_dengue_l.drop(columns=['Latitude', 'Longitude']).copy()
df_dengue_m

,idx_city,week,count
0,COLOMBIA_AMAZONAS_LETICIA,2023-01-02,7.0
1,COLOMBIA_AMAZONAS_LETICIA,2023-01-09,11.0
2,COLOMBIA_AMAZONAS_LETICIA,2023-01-16,5.0
3,COLOMBIA_AMAZONAS_LETICIA,2023-01-23,8.0
4,COLOMBIA_AMAZONAS_LETICIA,2023-01-30,17.0
...,...,...,...
40919,COLOMBIA_RISARALDA_BELEN DE UMBRIA,2023-11-27,0.0
40920,COLOMBIA_RISARALDA_BELEN DE UMBRIA,2023-12-04,0.0
40921,COLOMBIA_RISARALDA_BELEN DE UMBRIA,2023-12-11,0.0
40922,COLOMBIA_RISARALDA_BELEN DE UMBRIA,2023-12-18,0.0


In [47]:
df_dengue_again = df_dengue_m.merge(df_cities, how='left', on = 'idx_city')
df_dengue_again

,idx_city,week,count,Latitude,Longitude
0,COLOMBIA_AMAZONAS_LETICIA,2023-01-02,7.0,-4.212921,-69.942596
1,COLOMBIA_AMAZONAS_LETICIA,2023-01-09,11.0,-4.212921,-69.942596
2,COLOMBIA_AMAZONAS_LETICIA,2023-01-16,5.0,-4.212921,-69.942596
3,COLOMBIA_AMAZONAS_LETICIA,2023-01-23,8.0,-4.212921,-69.942596
4,COLOMBIA_AMAZONAS_LETICIA,2023-01-30,17.0,-4.212921,-69.942596
...,...,...,...,...,...
40919,COLOMBIA_RISARALDA_BELEN DE UMBRIA,2023-11-27,0.0,5.200909,-75.868993
40920,COLOMBIA_RISARALDA_BELEN DE UMBRIA,2023-12-04,0.0,5.200909,-75.868993
40921,COLOMBIA_RISARALDA_BELEN DE UMBRIA,2023-12-11,0.0,5.200909,-75.868993
40922,COLOMBIA_RISARALDA_BELEN DE UMBRIA,2023-12-18,0.0,5.200909,-75.868993


# Weather

In [48]:
weather_weekly_path = 'meteo_colombia/cities_weekly/'
files_csv = os.listdir(weather_weekly_path)
files_csv

['ARUBA_EXTERIOR_EXTERIOR_ARUBA.csv',
 'COLOMBIA_AMAZONAS_LETICIA.csv',
 'COLOMBIA_ANTIOQUIA_AMAGA.csv',
 'COLOMBIA_ANTIOQUIA_ANTIOQUIA.csv',
 'COLOMBIA_ANTIOQUIA_ANZA.csv',
 'COLOMBIA_ANTIOQUIA_APARTADO.csv',
 'COLOMBIA_ANTIOQUIA_BELLO.csv',
 'COLOMBIA_ANTIOQUIA_BOLIVAR.csv',
 'COLOMBIA_ANTIOQUIA_CALDAS.csv',
 'COLOMBIA_ANTIOQUIA_CAREPA.csv',
 'COLOMBIA_ANTIOQUIA_CARMEN DE VIBORAL.csv',
 'COLOMBIA_ANTIOQUIA_CHIGORODO.csv',
 'COLOMBIA_ANTIOQUIA_COCORNA.csv',
 'COLOMBIA_ANTIOQUIA_COPACABANA.csv',
 'COLOMBIA_ANTIOQUIA_EBEJICO.csv',
 'COLOMBIA_ANTIOQUIA_ENVIGADO.csv',
 'COLOMBIA_ANTIOQUIA_GIRARDOTA.csv',
 'COLOMBIA_ANTIOQUIA_GUATAPE.csv',
 'COLOMBIA_ANTIOQUIA_ITAGUI.csv',
 'COLOMBIA_ANTIOQUIA_LA CEJA.csv',
 'COLOMBIA_ANTIOQUIA_LA ESTRELLA.csv',
 'COLOMBIA_ANTIOQUIA_MEDELLIN.csv',
 'COLOMBIA_ANTIOQUIA_RIONEGRO.csv',
 'COLOMBIA_ANTIOQUIA_SABANETA.csv',
 'COLOMBIA_ANTIOQUIA_SAN JERONIMO.csv',
 'COLOMBIA_ANTIOQUIA_SANTA BARBARA.csv',
 'COLOMBIA_ANTIOQUIA_TURBO.csv',
 'COLOMBIA_ANTIOQUIA_VENEC

In [49]:
df_weather = pd.DataFrame()
for file in files_csv:
    df_temp = pd.read_csv(weather_weekly_path + file)
    df_temp['idx'] = file.split('.')[0]
    df_weather = pd.concat([df_weather, df_temp], ignore_index=True)

df_weather.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14310 entries, 0 to 14309
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   week    14310 non-null  object 
 1   tavg    14310 non-null  float64
 2   tmin    14310 non-null  float64
 3   tmax    14310 non-null  float64
 4   prcp    14310 non-null  float64
 5   wdir    14310 non-null  float64
 6   wspd    14310 non-null  float64
 7   pres    14246 non-null  float64
 8   idx     14310 non-null  object 
dtypes: float64(7), object(2)
memory usage: 1006.3+ KB


In [50]:
df_weather['week'] = pd.to_datetime(df_weather['week'])
df_weather['week'] = df_weather['week'].dt.to_period('W').apply(lambda r: r.start_time)

In [51]:
df_weather

,week,tavg,tmin,tmax,prcp,wdir,wspd,pres,idx
0,2022-12-26,26.800000,24.500000,30.000000,0.700000,82.000000,23.300000,1014.000000,ARUBA_EXTERIOR_EXTERIOR_ARUBA
1,2023-01-02,26.685714,24.400000,29.785714,0.814286,87.000000,23.242857,1014.271429,ARUBA_EXTERIOR_EXTERIOR_ARUBA
2,2023-01-09,26.742857,24.600000,29.671429,0.157143,93.000000,20.942857,1013.457143,ARUBA_EXTERIOR_EXTERIOR_ARUBA
3,2023-01-16,26.628571,24.514286,29.528571,0.242857,93.000000,25.271429,1013.785714,ARUBA_EXTERIOR_EXTERIOR_ARUBA
4,2023-01-23,26.385714,24.471429,28.914286,0.614286,92.428571,26.428571,1014.071429,ARUBA_EXTERIOR_EXTERIOR_ARUBA
...,...,...,...,...,...,...,...,...,...
14305,2023-11-27,16.985714,13.971429,20.871429,9.028571,136.428571,7.800000,1015.157143,INDIA_EXTERIOR_EXTERIOR_INDIA
14306,2023-12-04,16.842857,13.385714,20.314286,0.257143,75.428571,8.500000,1014.814286,INDIA_EXTERIOR_EXTERIOR_INDIA
14307,2023-12-11,16.057143,9.200000,20.728571,0.000000,108.857143,6.785714,1016.500000,INDIA_EXTERIOR_EXTERIOR_INDIA
14308,2023-12-18,15.000000,8.114286,21.271429,0.000000,76.000000,8.014286,1018.628571,INDIA_EXTERIOR_EXTERIOR_INDIA


In [52]:
df_merge = df_dengue_again.merge(df_weather, left_on=['idx_city', 'week'], right_on=['idx', 'week'], how='left')

In [53]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40924 entries, 0 to 40923
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   idx_city   40924 non-null  object        
 1   week       40924 non-null  datetime64[ns]
 2   count      40924 non-null  float64       
 3   Latitude   40508 non-null  float64       
 4   Longitude  40508 non-null  float64       
 5   tavg       13572 non-null  float64       
 6   tmin       13572 non-null  float64       
 7   tmax       13572 non-null  float64       
 8   prcp       13572 non-null  float64       
 9   wdir       13572 non-null  float64       
 10  wspd       13572 non-null  float64       
 11  pres       13508 non-null  float64       
 12  idx        13572 non-null  object        
dtypes: datetime64[ns](1), float64(10), object(2)
memory usage: 4.1+ MB


In [54]:
df_merge.drop(columns=['idx'], inplace=True)

In [55]:
df_merge.reset_index(drop=True, inplace=True)

In [56]:
df_merge.columns

Index(['idx_city', 'week', 'count', 'Latitude', 'Longitude', 'tavg', 'tmin',
       'tmax', 'prcp', 'wdir', 'wspd', 'pres'],
      dtype='object')

In [57]:
df_merge_2 = df_merge.merge(df_elevation, left_on='idx_city', right_on='idx_city', how='left')

In [58]:
df_merge_nans = df_merge_2.copy()

# MVP

In [59]:
df_merge_2.dropna(subset=['tavg'], inplace=True)

In [60]:
df_merge_2.head()

,idx_city,week,count,Latitude,Longitude,tavg,tmin,tmax,prcp,wdir,wspd,pres,elevation
0,COLOMBIA_AMAZONAS_LETICIA,2023-01-02,7.0,-4.212921,-69.942596,25.228571,22.985714,29.171429,8.528571,56.000000,4.800000,1010.985714,81.0
1,COLOMBIA_AMAZONAS_LETICIA,2023-01-09,11.0,-4.212921,-69.942596,25.000000,22.328571,29.042857,12.128571,207.857143,4.228571,1010.957143,81.0
2,COLOMBIA_AMAZONAS_LETICIA,2023-01-16,5.0,-4.212921,-69.942596,24.942857,22.257143,28.714286,12.971429,118.857143,3.800000,1010.814286,81.0
3,COLOMBIA_AMAZONAS_LETICIA,2023-01-23,8.0,-4.212921,-69.942596,25.657143,22.771429,30.471429,12.671429,200.857143,4.200000,1009.657143,81.0
4,COLOMBIA_AMAZONAS_LETICIA,2023-01-30,17.0,-4.212921,-69.942596,24.871429,22.328571,29.228571,16.728571,111.714286,4.028571,1010.857143,81.0


In [61]:
# create dir if not os.path.exists('silver'):
if not os.path.exists('platinum'):
    os.makedirs('platinum')

In [62]:
df_merge_2.to_csv('platinum/dengue_weather.csv', index=False)

In [71]:
df_merge_2.drop(columns=['week']).describe()

,count,Latitude,Longitude,tavg,tmin,tmax,prcp,wdir,wspd,pres,elevation
count,13572.000000,13572.000000,13572.000000,13572.000000,13572.000000,13572.000000,13572.000000,13572.000000,13572.000000,13508.000000,13468.000000
mean,4.677203,5.821634,-75.040179,24.254569,19.817515,29.338894,6.835577,158.812755,8.710850,1013.557402,877.270270
std,21.396246,2.834317,1.633569,4.470054,4.776370,4.219239,7.086991,78.953336,3.478198,4.745784,810.018106
min,0.000000,-4.212921,-81.720416,11.000000,3.457143,13.371429,0.000000,6.714286,2.328571,1006.542857,0.000000
25%,0.000000,4.177615,-75.768505,22.528571,17.442857,27.985714,1.628571,93.428571,6.457143,1010.128571,171.000000
50%,0.000000,5.023475,-75.243118,25.300000,21.314286,30.128571,5.042857,156.714286,8.228571,1012.142857,553.000000
75%,2.000000,7.831774,-74.271502,27.328571,23.400000,32.132143,9.857143,219.285714,10.014286,1015.657143,1479.000000
max,456.000000,12.582760,-67.484189,33.200000,27.957143,37.528571,61.871429,351.285714,28.500000,1029.214286,3241.000000


In [72]:
latex_table = df_merge_2.drop(columns=['week']).describe().T.to_latex(
    caption="Descriptive statistics of the dataset.",
    label="tab:descriptive_statistics",
    float_format="%.2f",
    bold_rows=True,
    column_format="l" + "c" * len(df_merge_2.drop(columns=['week']).describe().columns),
    position="htbp"
)

print(latex_table)

\begin{table}[htbp]
\caption{Descriptive statistics of the dataset.}
\label{tab:descriptive_statistics}
\begin{tabular}{lccccccccccc}
\toprule
 & count & mean & std & min & 25% & 50% & 75% & max \\
\midrule
\textbf{count} & 13572.00 & 4.68 & 21.40 & 0.00 & 0.00 & 0.00 & 2.00 & 456.00 \\
\textbf{Latitude} & 13572.00 & 5.82 & 2.83 & -4.21 & 4.18 & 5.02 & 7.83 & 12.58 \\
\textbf{Longitude} & 13572.00 & -75.04 & 1.63 & -81.72 & -75.77 & -75.24 & -74.27 & -67.48 \\
\textbf{tavg} & 13572.00 & 24.25 & 4.47 & 11.00 & 22.53 & 25.30 & 27.33 & 33.20 \\
\textbf{tmin} & 13572.00 & 19.82 & 4.78 & 3.46 & 17.44 & 21.31 & 23.40 & 27.96 \\
\textbf{tmax} & 13572.00 & 29.34 & 4.22 & 13.37 & 27.99 & 30.13 & 32.13 & 37.53 \\
\textbf{prcp} & 13572.00 & 6.84 & 7.09 & 0.00 & 1.63 & 5.04 & 9.86 & 61.87 \\
\textbf{wdir} & 13572.00 & 158.81 & 78.95 & 6.71 & 93.43 & 156.71 & 219.29 & 351.29 \\
\textbf{wspd} & 13572.00 & 8.71 & 3.48 & 2.33 & 6.46 & 8.23 & 10.01 & 28.50 \\
\textbf{pres} & 13508.00 & 1013.56 & 4.75 &

In [75]:
len(df_merge_2['idx_city'].unique())

261

In [76]:
len(df_merge_2)

13572

# Data with NaNs

In [63]:
df_merge_nans.to_csv('silver/dengue_weather_nans.csv', index=False)